# Hacker News — Supplementary EDA

APAN 5205 Team Project — Supplementary Analysis

This notebook extends the main EDA with additional visualizations:
- Section 1: Monthly posting trends over time
- Section 2: Story type deep dive (title length, score distributions)
- Section 3: Author activity analysis
- Section 4: Text analysis (word frequencies + word clouds)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from wordcloud import WordCloud
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

sns.set_theme(style='whitegrid')
%matplotlib inline

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('data/hn_stories_clean.csv')
print('Shape:', df.shape)
df.head(3)

---
## Section 1 — Monthly Posting Trends

Our dataset spans 14 months (Feb 2025 – Mar 2026). Let's see how posting volume and average score changed over time. This helps verify data coverage and spot any seasonal patterns.

In [ ]:
monthly = df.groupby('year_month').agg(
    post_count=('score', 'count'),
    avg_score=('score', 'mean'),
    median_score=('score', 'median')
).reset_index()

monthly

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# post volume over time
axes[0].bar(monthly['year_month'], monthly['post_count'], color='steelblue', width=0.6)
axes[0].set_title('Monthly Post Volume (Feb 2025 – Mar 2026)', fontsize=13)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Number of Posts')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(monthly['post_count'].mean(), color='red', linestyle='--',
                label=f'avg = {monthly["post_count"].mean():.0f}')
axes[0].legend()

# avg score over time
axes[1].plot(monthly['year_month'], monthly['avg_score'],
             marker='o', color='coral', linewidth=2, label='mean score')
axes[1].plot(monthly['year_month'], monthly['median_score'],
             marker='s', color='goldenrod', linewidth=2, linestyle='--', label='median score')
axes[1].set_title('Monthly Average Score Over Time', fontsize=13)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# monthly breakdown by story type
monthly_type = df.groupby(['year_month', 'story_type']).size().unstack(fill_value=0)

# keep top 5 types for readability
top_types = df['story_type'].value_counts().head(5).index.tolist()
monthly_type[top_types].plot(
    kind='bar', figsize=(13, 5), width=0.8,
    colormap='tab10'
)
plt.title('Monthly Post Volume by Story Type (Top 5)', fontsize=13)
plt.xlabel('Month')
plt.ylabel('Number of Posts')
plt.xticks(rotation=45)
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

Post volume is fairly consistent across months (~1000 per month), which confirms our scraper collected data evenly across the time range. The dominant story type throughout is `external`, which makes sense — most HN posts are links to external articles.

---
## Section 2 — Story Type Deep Dive

We have 10 story types. Let's go beyond just counting posts and look at:
- Title length patterns across types
- Score distribution shape per type
- Whether `has_url` vs `has_text` differs by type

In [ ]:
# title length by story type (boxplot)
type_order = df.groupby('story_type')['title_length'].median().sort_values(ascending=False).index

plt.figure(figsize=(13, 5))
sns.boxplot(
    data=df,
    x='story_type', y='title_length',
    order=type_order,
    palette='Set2',
    showfliers=False
)
plt.title('Title Length Distribution by Story Type', fontsize=13)
plt.xlabel('Story Type')
plt.ylabel('Title Length (chars)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# log score distribution per story type (violin plot)
df['log_score'] = np.log1p(df['score'])

score_order = df.groupby('story_type')['log_score'].median().sort_values(ascending=False).index

plt.figure(figsize=(13, 5))
sns.violinplot(
    data=df,
    x='story_type', y='log_score',
    order=score_order,
    palette='Set3',
    inner='quartile'
)
plt.title('log1p(Score) Distribution by Story Type', fontsize=13)
plt.xlabel('Story Type')
plt.ylabel('log1p(Score)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# has_url vs has_text breakdown by story type
url_text = df.groupby('story_type')[['has_url', 'has_text']].mean().sort_values('has_url', ascending=False)

url_text.plot(kind='bar', figsize=(12, 4), color=['steelblue', 'coral'], width=0.6)
plt.title('Proportion of Posts with URL vs Body Text, by Story Type', fontsize=13)
plt.xlabel('Story Type')
plt.ylabel('Proportion')
plt.xticks(rotation=30, ha='right')
plt.legend(['has_url', 'has_text'])
plt.tight_layout()
plt.show()

Interesting patterns:
- `ask_hn` and `text_post` have longer titles on average — these posts tend to be more descriptive
- `research` and `news_major` consistently score higher across the distribution
- As expected, `external` posts almost always have a URL, while `ask_hn` / `text_post` have body text instead

---
## Section 3 — Author Activity Analysis

Who are the most active posters on HN? Do prolific posters get better scores, or does posting more mean lower average quality?

In [ ]:
author_stats = df.groupby('author').agg(
    post_count=('score', 'count'),
    avg_score=('score', 'mean'),
    median_score=('score', 'median'),
    total_score=('score', 'sum')
).sort_values('post_count', ascending=False)

print('Total unique authors:', len(author_stats))
print('\nTop 15 most active authors:')
author_stats.head(15).round(1)

In [ ]:
top15 = author_stats.head(15).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# post count
axes[0].barh(top15['author'][::-1], top15['post_count'][::-1], color='steelblue')
axes[0].set_title('Top 15 Authors by Post Count', fontsize=12)
axes[0].set_xlabel('Number of Posts')

# avg score for same authors
colors = ['mediumseagreen' if s >= author_stats['avg_score'].mean() else 'salmon'
          for s in top15['avg_score'][::-1]]
axes[1].barh(top15['author'][::-1], top15['avg_score'][::-1], color=colors)
axes[1].axvline(author_stats['avg_score'].mean(), color='black', linestyle='--',
                label=f'overall avg = {author_stats["avg_score"].mean():.1f}')
axes[1].set_title('Avg Score for Top 15 Authors', fontsize=12)
axes[1].set_xlabel('Mean Score')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# does posting more = higher or lower avg score?
# filter to authors with at least 5 posts for reliability
active_authors = author_stats[author_stats['post_count'] >= 5]

plt.figure(figsize=(8, 5))
plt.scatter(
    active_authors['post_count'],
    np.log1p(active_authors['avg_score']),
    alpha=0.3, s=15, color='steelblue'
)
plt.title('Post Count vs Avg Score (authors with ≥5 posts)', fontsize=12)
plt.xlabel('Number of Posts')
plt.ylabel('log1p(Avg Score)')
plt.tight_layout()
plt.show()

corr = active_authors['post_count'].corr(active_authors['avg_score'])
print(f'Correlation between post count and avg score: {corr:.4f}')

---
## Section 4 — Text Analysis

What words appear most frequently in HN titles? We'll look at overall word frequency and break it down by story type. We'll also generate word clouds for visual appeal.

In [ ]:
# text cleaning (same as clustering notebook)
stop_words = set(stopwords.words('english'))
# add HN-specific noise words
hn_noise = {'show', 'hn', 'ask', 'use', 'using', 'new', 'first',
            'get', 'make', 'us', 'go', 'would', 'built', 'via', 'year'}
stop_words |= hn_noise

def get_tokens(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return [w for w in text.split() if w not in stop_words and len(w) > 2]

all_tokens = df['title'].apply(get_tokens)
flat_tokens = [token for sublist in all_tokens for token in sublist]

word_freq = Counter(flat_tokens)
print('Total unique tokens:', len(word_freq))
print('\nTop 20 words:')
word_freq.most_common(20)

In [ ]:
# top 20 words bar chart
top20 = word_freq.most_common(20)
words, counts = zip(*top20)

plt.figure(figsize=(11, 5))
bars = plt.bar(words, counts, color='steelblue', edgecolor='white')
plt.title('Top 20 Most Frequent Words in HN Post Titles', fontsize=13)
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=35, ha='right')

# add count labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(count), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# overall word cloud
wordcloud = WordCloud(
    width=900, height=450,
    background_color='white',
    colormap='Blues',
    max_words=150,
    collocations=False
).generate_from_frequencies(word_freq)

plt.figure(figsize=(13, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — All HN Post Titles', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# word clouds per story type (top 4 types)
top4_types = df['story_type'].value_counts().head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, stype in zip(axes, top4_types):
    subset = df[df['story_type'] == stype]['title']
    tokens = [t for title in subset for t in get_tokens(title)]
    freq = Counter(tokens)
    
    wc = WordCloud(
        width=500, height=300,
        background_color='white',
        max_words=80,
        collocations=False
    ).generate_from_frequencies(freq)
    
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{stype} (n={len(subset)})', fontsize=11)

plt.suptitle('Word Clouds by Story Type (Top 4)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# top 10 words per story type -- heatmap style
type_word_freq = {}
for stype in df['story_type'].unique():
    subset = df[df['story_type'] == stype]['title']
    tokens = [t for title in subset for t in get_tokens(title)]
    type_word_freq[stype] = Counter(tokens)

# get top 15 overall words and see their freq per type
top15_words = [w for w, _ in word_freq.most_common(15)]

heatmap_data = pd.DataFrame(
    {stype: [type_word_freq[stype].get(w, 0) for w in top15_words]
     for stype in df['story_type'].unique()},
    index=top15_words
)

# normalize by story type total posts for fair comparison
type_counts = df['story_type'].value_counts()
heatmap_norm = heatmap_data.div(type_counts, axis=1) * 1000  # per 1000 posts

plt.figure(figsize=(13, 6))
sns.heatmap(heatmap_norm, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.5, cbar_kws={'label': 'Occurrences per 1000 posts'})
plt.title('Top 15 Words Frequency by Story Type (normalized)', fontsize=13)
plt.xlabel('Story Type')
plt.ylabel('Word')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## Summary

**Monthly trends:**
- Post volume is consistent (~1000/month), confirming even data coverage across 14 months
- Average score fluctuates month to month but shows no strong seasonal pattern

**Story type deep dive:**
- `ask_hn` and `text_post` tend to have longer titles
- `research` and `news_major` score highest across the distribution
- URL vs text content cleanly separates story types as expected

**Author analysis:**
- A small number of prolific authors dominate post volume
- No strong relationship between posting frequency and average score — quantity doesn't guarantee quality

**Text analysis:**
- Most common words reflect HN's tech-focused community (open, source, software, web, python)
- Different story types have distinctly different vocabularies, visible in both word clouds and the frequency heatmap